# Импорты

In [17]:
import math
import random
from typing import List, Tuple, Callable

from scipy import stats
import pandas as pd
import numpy as np

# 1. Реализация метода bootstrap

Метод bootstrap: <br>
Имеется выборка $X_1, \ldots, X_n$ из $X$ <br>
$\alpha -$ уровень доверия <br>
Алгоритм вычисления доверительного интервала параметра $\theta$ <br>
1) Найти оценку параметра $\^{\theta}_n$
2) Сгенерировать $k$ новых выборок $X_1^i, \ldots, X_n^i, ~~ i = \overline{1, k}, ~~ X_j^i -$ случайный элемент исходной выборки
3) $\theta_i^* = \^{\theta}_n (X_1^i, \ldots, X_n^i), ~~ i = \overline{1, k}$
4) Доверительный интервал $- \displaystyle \left( \theta_{\scriptsize{\displaystyle \left\lfloor \frac{1 - \alpha}{2} k \right\rfloor}}^*; \theta_{\scriptsize{\displaystyle \left\lceil \frac{1 + \alpha}{2} k \right\rceil}}^* \right)$

In [18]:
def bootstrap(X_n: List[float], theta_n: Callable[[List[float]], float], alpha: float = 0.05) -> Tuple[float, float]:
    k = 1000
    theta_i = [0] * k
    
    for i in range(k):
        X_i = random.choices(X_n, k=len(X_n))
        theta_i[i] = theta_n(X_i)
    theta_i.sort()
    
    left = math.floor((1 - alpha) / 2 * k)
    right = math.ceil((1 + alpha) / 2 * k)
    
    return theta_i[left], theta_i[right]

# 2. Сравнение собственного метода bootstrap с реализацией из scipy.stats

In [19]:
random_state = 18
data = {
    "my": {
        "uniform": {100: {}, 1000: {}},
        "bernoulli": {100: {}, 1000: {}},
        "binom": {100: {}, 1000: {}},
        "norm": {100: {}, 1000: {}}
    },
    "scipy": {
        "uniform": {100: {}, 1000: {}},
        "bernoulli": {100: {}, 1000: {}},
        "binom": {100: {}, 1000: {}},
        "norm": {100: {}, 1000: {}}
    }
}

## Равномерное распределение

$\displaystyle
X \sim U(a, b) \\
X_1, \ldots, X_n - \text{ выборка из } X \\
F_X \in \{F(x, a, b) ~|~ a, b \in \mathbb{R}, b \geq a\} \\
f_X(x) = \frac{I(x \in [a; b])}{(b - a)^n} \\
L(a, b, X_1, \ldots, X_n) = \prod_{i = 1}^n \frac{I(X_i \in [a; b])}{(b - a)^n} = \frac{1}{(b - a)^n} \prod_{i = 1}^n I(X_i \in [a; b]) \\
\^{a}_n = \argmax_{a \leq b} \frac{1}{(b - a)^n} \prod_{i = 1}^n I(X_i \in [a; b]) = \argmax_{a \leq X_{\min}} \frac{1}{(b - a)^n} = X_{\min} \\
\^{b}_n = \argmax_{b \geq a} \frac{1}{(b - a)^n} \prod_{i = 1}^n I(X_i \in [a; b]) = \argmax_{b \geq X_{\max}} \frac{1}{(b - a)^n} = X_{\max} \\
\text{Оценка параметра } \^{a}_n = X_{\min} \\
\text{Оценка параметра } \^{b}_n = X_{\max} \\
$

In [20]:
def a_n_U(X_n: List[float]) -> float:
    return min(X_n)
def b_n_U(X_n: List[float]) -> float:
    return max(X_n)


sample_100 = stats.uniform.rvs(loc=-1, scale=5, size=100, random_state=random_state)


data["my"]["uniform"][100]["a_n"] = bootstrap(
    X_n=sample_100,
    theta_n=a_n_U
)
data["my"]["uniform"][100]["b_n"] = bootstrap(
    X_n=sample_100,
    theta_n=b_n_U
)

res = stats.bootstrap(
    data=(sample_100,),
    statistic=a_n_U,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["uniform"][100]["a_n"] = (
    res.confidence_interval.low, 
    res.confidence_interval.high
)
res = stats.bootstrap(
    data=(sample_100,),
    statistic=b_n_U,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["uniform"][100]["b_n"] = (
    res.confidence_interval.low, 
    res.confidence_interval.high
)



sample_1000 = stats.uniform.rvs(loc=-1, scale=5, size=100, random_state=random_state)


data["my"]["uniform"][1000]["a_n"] = bootstrap(
    X_n=sample_1000,
    theta_n=a_n_U
)
data["my"]["uniform"][1000]["b_n"] = bootstrap(
    X_n=sample_1000,
    theta_n=b_n_U
)

res = stats.bootstrap(
    data=(sample_1000,),
    statistic=a_n_U,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["uniform"][1000]["a_n"] = (
    res.confidence_interval.low, 
    res.confidence_interval.high
)
res = stats.bootstrap(
    data=(sample_1000,),
    statistic=b_n_U,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["uniform"][1000]["b_n"] = (
    res.confidence_interval.low, 
    res.confidence_interval.high
)



print(data["my"]["uniform"])
print(data["scipy"]["uniform"])

{100: {'a_n': (np.float64(-0.9449165257300504), np.float64(-0.9449165257300504)), 'b_n': (np.float64(3.9869264123982333), np.float64(3.9869264123982333))}, 1000: {'a_n': (np.float64(-0.9449165257300504), np.float64(-0.9449165257300504)), 'b_n': (np.float64(3.9869264123982333), np.float64(3.9869264123982333))}}
{100: {'a_n': (np.float64(-0.9449165257300504), np.float64(-0.9383373125853969)), 'b_n': (np.float64(3.942489380219624), np.float64(3.9869264123982333))}, 1000: {'a_n': (np.float64(-0.9449165257300504), np.float64(-0.9383373125853969)), 'b_n': (np.float64(3.942489380219624), np.float64(3.9869264123982333))}}


## Распределение Бернулли

$\displaystyle
X \sim Bernoulli(p) \\
X_1, \ldots, X_n - \text{ выборка из } X \\
F_X \in \{F_X(x, p) ~|~ p \in [0; 1] \} \\
p_X(x) = p^x (1 - p)^{1 - x} \\
L(p, X_1, \ldots, X_n) = \prod_{i = 1}^n p^{X_i} (1 - p)^{1 - X_i} = p^{\scriptsize \displaystyle \sum_{i = 1}^n X_i} (1 - p)^{\scriptsize \displaystyle n - \sum_{i = 1}^n X_i} \\
\ln L(p, X_1, \ldots, X_n) = \ln \left( p^{\scriptsize \displaystyle \sum_{i = 1}^n X_i} (1 - p)^{\scriptsize \displaystyle n - \sum_{i = 1}^n X_i} \right) = \sum_{i = 1}^n X_i \ln p + \left( n - \sum_{i = 1}^n X_i \right) \ln(1 - p) \\
\^{p}_n = \argmax_{p \in [0; 1]} \ln L(p, X_1, \ldots, X_n)
$

$\displaystyle
\frac{d\ln L}{dp} = \frac{\displaystyle \sum_{i = 1}^n X_i}{p} + \frac{\displaystyle \sum_{i = 1}^n X_i - n}{1 - p} = 0 \\
(1 - p) \sum_{i = 1}^n X_i + \left( \sum_{i = 1}^n X_i - n \right) p = 0 \\
pn = \sum_{i = 1}^n X_i \\
p = \sum_{i = 1}^n \frac{X_i}{n} \\
\frac{d^2 \ln L}{dp^2} = \underbrace{-\frac{\displaystyle \sum_{i = 1}^n X_i}{p^2}}_{\displaystyle \leq 0} + \underbrace{\frac{\displaystyle \sum_{i = 1}^n X_i - n}{(1 - p)^2}}_{\displaystyle \leq 0} \leq 0
$

$\displaystyle
\text{Оценка параметра } \^{p}_n = \sum_{i = 1}^n \frac{X_i}{n} = \overline{X}
$

In [21]:
def p_n_Bernoulli(X_n: List[float]) -> float:
    return sum(X_n) / len(X_n)


sample_100 = stats.bernoulli.rvs(p=0.25, size=100, random_state=random_state)


data["my"]["bernoulli"][100] = bootstrap(
    X_n=sample_100,
    theta_n=p_n_Bernoulli
)

res = stats.bootstrap(
    data=(sample_100,),
    statistic=p_n_Bernoulli,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["bernoulli"][100] = (
    res.confidence_interval.low,
    res.confidence_interval.high
)



sample_1000 = stats.bernoulli.rvs(p=0.25, size=1000, random_state=random_state)


data["my"]["bernoulli"][1000] = bootstrap(
    X_n=sample_1000,
    theta_n=p_n_Bernoulli
)

res = stats.bootstrap(
    data=(sample_1000,),
    statistic=p_n_Bernoulli,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["bernoulli"][1000] = (
    res.confidence_interval.low,
    res.confidence_interval.high
)



print(data["my"]["bernoulli"])
print(data["scipy"]["bernoulli"])

{100: (np.float64(0.27), np.float64(0.27)), 1000: (np.float64(0.242), np.float64(0.244))}
{100: (np.float64(0.19), np.float64(0.36)), 1000: (np.float64(0.21599999999999997), np.float64(0.26926529518681824))}


## Биномиальное распределение

$\displaystyle
X \sim Bin(k, p) \\
X_1, \ldots, X_n - \text{ выборка из } X \\
F_X \in \{F_X(x, k, p) ~|~ p \in [0; 1], k \in \mathbb{N} \} \\
\begin{cases}
    E(X) = \overline{X} \\
    D(X) = \overline{X^2} - \overline{X}^2
\end{cases} \Leftrightarrow \begin{cases}
    kp = \overline{X} \\
    kp(1 - p) = \overline{X^2} - \overline{X}^2
\end{cases} \Leftrightarrow \begin{cases}
    \displaystyle p = \frac{\overline{X} - \overline{X^2} + \overline{X}^2}{\overline{X}} \\
    \displaystyle k = \frac{\overline{X}^2}{\overline{X} - \overline{X^2} + \overline{X}^2}
\end{cases} \\
\text{Оценка параметра } \^{p}_n = \frac{\overline{X} - \overline{X^2} + \overline{X}^2}{\overline{X}} \\
\text{Оценка параметра } \^{k}_n = \frac{\overline{X}^2}{\overline{X} - \overline{X^2} + \overline{X}^2}
$

In [22]:
def p_n_Bin(X_n: List[float]) -> float:
    X_mean = sum(X_n) / len(X_n)
    X_2_mean = sum(X ** 2 for X in X_n) / len(X_n)

    return (X_mean - X_2_mean + X_mean ** 2) / X_mean

def k_n_Bin(X_n: List[float]) -> float:
    X_mean = sum(X_n) / len(X_n)
    X_2_mean = sum(X ** 2 for X in X_n) / len(X_n)

    return X_mean ** 2 / (X_mean - X_2_mean + X_mean ** 2)


sample_100 = stats.binom.rvs(p=0.25, n=17, size=100, random_state=random_state)


data["my"]["binom"][100]["p_n"] = bootstrap(
    X_n=sample_100,
    theta_n=p_n_Bin
)
data["my"]["binom"][100]["k_n"] = bootstrap(
    X_n=sample_100,
    theta_n=k_n_Bin
)

res = stats.bootstrap(
    data=(sample_100,),
    statistic=p_n_Bin,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["binom"][100]["p_n"] = (
    res.confidence_interval.low, 
    res.confidence_interval.high
)
res = stats.bootstrap(
    data=(sample_100,),
    statistic=k_n_Bin,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["binom"][100]["k_n"] = (
    res.confidence_interval.low, 
    res.confidence_interval.high
)


sample_1000 = stats.binom.rvs(p=0.25, n=17, size=1000, random_state=random_state)


data["my"]["binom"][1000]["p_n"] = bootstrap(
    X_n=sample_1000,
    theta_n=p_n_Bin
)
data["my"]["binom"][1000]["k_n"] = bootstrap(
    X_n=sample_1000,
    theta_n=k_n_Bin
)

res = stats.bootstrap(
    data=(sample_1000,),
    statistic=p_n_Bin,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["binom"][1000]["p_n"] = (
    res.confidence_interval.low, 
    res.confidence_interval.high
)
res = stats.bootstrap(
    data=(sample_1000,),
    statistic=k_n_Bin,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["binom"][1000]["k_n"] = (
    res.confidence_interval.low, 
    res.confidence_interval.high
)



print(data["my"]["binom"])
print(data["scipy"]["binom"])

{100: {'p_n': (np.float64(0.14824228028503614), np.float64(0.16695550351288002)), 'k_n': (np.float64(21.225859662479472), np.float64(22.914175506268005))}, 1000: {'p_n': (np.float64(0.2300248328557774), np.float64(0.23468406337371756)), 'k_n': (np.float64(18.257301704275168), np.float64(18.53864355340651))}}
{100: {'p_n': (np.float64(-0.11928627669369297), np.float64(0.3526027397260273)), 'k_n': (np.float64(10.723034762466831), np.float64(1092.8515993402768))}, 1000: {'p_n': (np.float64(0.14938975414798977), np.float64(0.28936995610249944)), 'k_n': (np.float64(14.627559900022698), np.float64(28.749774846896145))}}


## Нормальное распределение

$\displaystyle
X \sim N(\mu, \sigma^2) \\
X_1, \ldots, X_n - \text{ выборка из } X \\
F_X \in \{ F_X(x, \mu, \sigma^2) ~|~ \mu, \sigma \in \mathbb{R} \} \\
\begin{cases}
    E(X) = \overline{X} \\
    D(X) = \overline{X^2} - \overline{X}^2
\end{cases} \Leftrightarrow \begin{cases}
    \mu = \overline{X} \\
    \sigma^2 = \overline{X^2} - \overline{X}^2
\end{cases} \Leftrightarrow \begin{cases}
    \mu = \overline{X} \\
    \sigma = \sqrt{\overline{X^2} - \overline{X}^2}
\end{cases} \\
\text{Оценка параметра } \^{\mu}_n = \overline{X} \\
\text{Оценка параметра } \^{\sigma}_n = \sqrt{\overline{X^2} - \overline{X}^2}
$

In [23]:
def mu_n_N(X_n: List[float]) -> float:
    return sum(X_n) / len(X_n)

def sigma_n_N(X_n: List[float]) -> float:
    X_mean = sum(X_n) / len(X_n)
    X_2_mean = sum(X ** 2 for X in X_n) / len(X_n)

    return math.sqrt(X_2_mean - X_mean ** 2)


sample_100 = stats.norm.rvs(loc=15, scale=5, size=100, random_state=random_state)


data["my"]["norm"][100]["mu_n"] = bootstrap(
    X_n=sample_100,
    theta_n=mu_n_N
)
data["my"]["norm"][100]["sigma_n"] = bootstrap(
    X_n=sample_100,
    theta_n=sigma_n_N
)

res = stats.bootstrap(
    data=(sample_100,),
    statistic=mu_n_N,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["norm"][100]["mu_n"] = (
    res.confidence_interval.low, 
    res.confidence_interval.high
)
res = stats.bootstrap(
    data=(sample_100,),
    statistic=sigma_n_N,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["norm"][100]["sigma_n"] = (
    res.confidence_interval.low, 
    res.confidence_interval.high
)


sample_1000 = stats.norm.rvs(loc=15, scale=5, size=1000, random_state=random_state)


data["my"]["norm"][1000]["mu_n"] = bootstrap(
    X_n=sample_1000,
    theta_n=mu_n_N
)
data["my"]["norm"][1000]["sigma_n"] = bootstrap(
    X_n=sample_1000,
    theta_n=sigma_n_N
)

res = stats.bootstrap(
    data=(sample_1000,),
    statistic=mu_n_N,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["norm"][1000]["mu_n"] = (
    res.confidence_interval.low, 
    res.confidence_interval.high
)
res = stats.bootstrap(
    data=(sample_1000,),
    statistic=sigma_n_N,
    vectorized=False,
    confidence_level=0.95,
    n_resamples=1000,
    random_state=random_state
)
data["scipy"]["norm"][1000]["sigma_n"] = (
    res.confidence_interval.low, 
    res.confidence_interval.high
)



print(data["my"]["norm"])
print(data["scipy"]["norm"])

{100: {'mu_n': (np.float64(15.874683048367567), np.float64(15.927964324497525)), 'sigma_n': (5.46365150664317, 5.510355704565014)}, 1000: {'mu_n': (np.float64(14.94984442509457), np.float64(14.973791432680871)), 'sigma_n': (5.178524780360844, 5.191796260985638)}}
{100: {'mu_n': (np.float64(14.91708098128866), np.float64(17.092283508916307)), 'sigma_n': (np.float64(4.776694841695162), np.float64(6.253489401407557))}, 1000: {'mu_n': (np.float64(14.635815965428733), np.float64(15.277720167120501)), 'sigma_n': (np.float64(4.980095941390632), np.float64(5.443105765828511))}}


# 3. Табличный результат

In [32]:
index = [
    ("Равномерное", "a", -1, 100),
    ("Равномерное", "a", -1, 1000),
    ("Равномерное", "b", 4, 100),
    ("Равномерное", "b", 4, 1000),

    ("Бернулли", "p", 0.25, 100),
    ("Бернулли", "p", 0.25, 1000),

    ("Биномиальное", "k", 17, 100),
    ("Биномиальное", "k", 17, 1000),
    ("Биномиальное", "p", 0.25, 100),
    ("Биномиальное", "p", 0.25, 1000),

    ("Нормальное", "mu", 15, 100),
    ("Нормальное", "mu", 15, 1000),
    ("Нормальное", "sigma", 5, 100),
    ("Нормальное", "sigma", 5, 1000)
]
multi_index = pd.MultiIndex.from_tuples(index, names=["Распределение", "Параметр", "Истинное значение", "Размер выборки"])

intervals = [
    [   # "Равномерное", "a", -1, 100
        data["my"]["uniform"][100]["a_n"], 
        data["scipy"]["uniform"][100]["a_n"]
    ],
    [   # "Равномерное", "a", -1, 1000
        data["my"]["uniform"][1000]["a_n"], 
        data["scipy"]["uniform"][1000]["a_n"]
    ],
    [   # "Равномерное", "b", 4, 100
        data["my"]["uniform"][100]["b_n"],
        data["scipy"]["uniform"][100]["b_n"]
    ],
    [   # "Равномерное", "b", 4, 1000
        data["my"]["uniform"][1000]["b_n"],
        data["scipy"]["uniform"][1000]["b_n"]
    ],

    [   # "Бернулли", "p", 0.25, 100
        data["my"]["bernoulli"][100], 
        data["scipy"]["bernoulli"][100]
    ],
    [   # "Бернулли", "p", 0.25, 1000
        data["my"]["bernoulli"][1000], 
        data["scipy"]["bernoulli"][1000]
    ],

    [   # "Биномиальное", "k", 17, 100
        data["my"]["binom"][100]["k_n"],
        data["scipy"]["binom"][100]["k_n"]
    ],
    [   # "Биномиальное", "k", 17, 1000
        data["my"]["binom"][1000]["k_n"],
        data["scipy"]["binom"][1000]["k_n"]
    ],
    [   # "Биномиальное", "p", 0.25, 100
        data["my"]["binom"][100]["p_n"],
        data["scipy"]["binom"][100]["p_n"]
    ],
    [   # "Биномиальное", "p", 0.25, 1000
        data["my"]["binom"][1000]["p_n"],
        data["scipy"]["binom"][1000]["p_n"]
    ],

    [   # "Нормальное", "mu", 15, 100
        data["my"]["norm"][100]["mu_n"],
        data["scipy"]["norm"][100]["mu_n"]
    ],
    [   # "Нормальное", "mu", 15, 1000
        data["my"]["norm"][1000]["mu_n"],
        data["scipy"]["norm"][1000]["mu_n"]
    ],
    [   # "Нормальное", "sigma", 5, 100
        data["my"]["norm"][100]["sigma_n"],
        data["scipy"]["norm"][100]["sigma_n"]
    ],
    [   # "Нормальное", "sigma", 5, 100
        data["my"]["norm"][1000]["sigma_n"],
        data["scipy"]["norm"][1000]["sigma_n"]
    ],
]

df = pd.DataFrame(intervals, index=multi_index, columns=["Моя", "SciPy"])
pd.set_option('display.expand_frame_repr', False) 
pd.set_option('display.max_colwidth', None)

df

Моя                                       SciPy
Распределение Параметр Истинное значение Размер выборки                                                                                        
Равномерное   a        -1.00             100             (-0.9449165257300504, -0.9449165257300504)  (-0.9449165257300504, -0.9383373125853969)
                                         1000            (-0.9449165257300504, -0.9449165257300504)  (-0.9449165257300504, -0.9383373125853969)
              b         4.00             100               (3.9869264123982333, 3.9869264123982333)     (3.942489380219624, 3.9869264123982333)
                                         1000              (3.9869264123982333, 3.9869264123982333)     (3.942489380219624, 3.9869264123982333)
Бернулли      p         0.25             100                                           (0.27, 0.27)                                (0.19, 0.36)
                                         1000                                        (0.242, 0.244)  (0.21599999999999997, 0.26926529518681824)
Биномиальное  k         17.00            100               (21.225859662479472, 22.914175506268005)    (10.723034762466831, 1092.8515993402768)
                                         1000               (18.257301704275168, 18.53864355340651)    (14.627559900022698, 28.749774846896145)
              p         0.25             100             (0.14824228028503614, 0.16695550351288002)  (-0.11928627669369297, 0.3526027397260273)
                                         1000             (0.2300248328557774, 0.23468406337371756)  (0.14938975414798977, 0.28936995610249944)
Нормальное    mu        15.00            100               (15.874683048367567, 15.927964324497525)     (14.91708098128866, 17.092283508916307)
                                         1000               (14.94984442509457, 14.973791432680871)    (14.635815965428733, 15.277720167120501)
              sigma     5.00             100                  (5.46365150664317, 5.510355704565014)      (4.776694841695162, 6.253489401407557)
                                         1000                (5.178524780360844, 5.191796260985638)      (4.980095941390632, 5.443105765828511)